# Experimentos — ronda 2 · AICOLOMBIA

La ronda 1 mostró que **ninguna variante superó a la V1** y que **menos árboles rinden mejor**: el clasificador se
sobreajusta rápido porque la señal está casi toda en el historial. Esta ronda prueba modelos más simples, pesos por
año, contexto espacial y un ensamble, siempre **contra la V1**.

| Variante | Qué cambia respecto a la V1 |
|---|---|
| `V1_early_stopping`, `E1_arboles_200`, `E4_hojas_15` | Se vuelven a entrenar (iguales a la ronda 1): comprueban que los resultados se reproducen y alimentan el ensamble |
| `E5_arboles_50` / `E5_arboles_100` | Clasificador con 50 / 100 árboles fijos |
| `E6_simple` | Clasificador con 100 árboles, 15 hojas y mínimo 200 filas por hoja |
| `E7_lento` | Tasa de aprendizaje 0.02 (pasos más cortos), con early stopping |
| `E8_peso_reciente` | Más peso a los años recientes (× 1.25 por año): el reporte crece cada año |
| `E10_espacial` | V1 + población (densidad × área), ubicación y tasa de los 5 municipios más cercanos |
| `E10_espacial_simple` | `E6_simple` + lo espacial de E10 |
| `E9_ensamble_top3` | Promedio de las 3 mejores variantes de esta ronda (probabilidades, y magnitud y cuantiles en log1p) |

⚠ **E10 usa cartografía municipal** (límites del MGN del DANE, vía la capa de la UPRA: ubicación y área). Antes de
**entregar** una variante E10, confirmar con los organizadores que la cartografía está permitida.

La tabla final junta **la ronda 1 y la ronda 2**. Regla para cambiar la entrega: mejorar ≥ 0.5 puntos a la V1 sin
empeorar en F3. Recuerden: una entrega cada 30 minutos; cuenta la última válida.

## 1. Datos, calidad y tratamiento (igual que antes)

In [1]:
import os, zipfile
import numpy as np
import pandas as pd

URL_DATOS = "PEGUE_AQUI_EL_ENLACE_DE_LOS_DATOS"  # enlace de Google Drive al archivo hackathon_datos.zip (se comparte el día del evento)
DATA_DIR = os.environ.get("HACKATHON_DATA", "datos")

if not os.path.exists(os.path.join(DATA_DIR, "entrenamiento.csv")):
    if URL_DATOS:
        import gdown
        import re
        ident = re.search(r"/d/([\w-]+)", URL_DATOS)   # enlace directo: funciona con cualquier versión de gdown
        url = f"https://drive.google.com/uc?id={ident.group(1)}" if ident else URL_DATOS
        gdown.download(url, "hackathon_datos.zip", quiet=False)
    with zipfile.ZipFile("hackathon_datos.zip") as z:
        z.extractall(DATA_DIR)

leer = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
train = leer("entrenamiento.csv")
test_pub = leer("prueba_equipos.csv")     # periodo de la tabla de posiciones en vivo
test_priv = leer("prueba_oculta.csv")     # periodo de la nota oficial
dic = pd.read_csv(os.path.join(DATA_DIR, "diccionario_datos.csv"))
print(train.shape, test_pub.shape, test_priv.shape)

(62814, 25) (20938, 13) (22040, 13)


In [2]:
ESPERADAS = {"entrenamiento": 62_814, "prueba_equipos": 20_938, "prueba_oculta": 22_040}
for nombre, df in {"entrenamiento": train, "prueba_equipos": test_pub, "prueba_oculta": test_priv}.items():
    print(f"{nombre:15s} filas={len(df):,} (esperadas {ESPERADAS[nombre]:,}) · municipios={df.DIVIPOLA.nunique()} · "
          f"llaves repetidas={df.duplicated(['DIVIPOLA', 'fecha']).sum()} · celdas nulas={int(df.isna().sum().sum())}")
nulos = pd.concat({"entrenamiento": train.isna().sum(), "prueba_equipos": test_pub.isna().sum(),
                   "prueba_oculta": test_priv.isna().sum()}, axis=1)
display(nulos[nulos.sum(axis=1) > 0])
fuera = pd.concat([test_pub, test_priv]).ONI.gt(train.ONI.max()).mean()
print(f"ONI de prueba por encima del máximo del entrenamiento: {fuera:.1%} de las filas (los árboles no extrapolan)")

entrenamiento   filas=62,814 (esperadas 62,814) · municipios=1102 · llaves repetidas=0 · celdas nulas=162
prueba_equipos  filas=20,938 (esperadas 20,938) · municipios=1102 · llaves repetidas=0 · celdas nulas=38
prueba_oculta   filas=22,040 (esperadas 22,040) · municipios=1102 · llaves repetidas=0 · celdas nulas=40


,entrenamiento,prueba_equipos,prueba_oculta
densidad_poblacional,12,0.0,0.0
ingresos_tributarios,36,0.0,0.0
inversion_gestion_riesgo,57,19.0,20.0
nbi_pct,57,19.0,20.0


ONI de prueba por encima del máximo del entrenamiento: 20.5% de las filas (los árboles no extrapolan)


In [3]:
COV_ESTATICAS = ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta"]
COV_ANUALES = ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]
RANGOS = {"densidad_poblacional": (0, None), "ingresos_tributarios": (0, None), "inversion_gestion_riesgo": (0, None),
          "nbi_pct": (0, 100), "elevacion_media": (-50, 6000), "pendiente_media": (0, 90),
          "pct_pendiente_alta": (0, 100), "ONI": (-3.5, 3.5)}
HISTORIAL = ["n_eventos", "familias_desplazadas", "viv_destruidas", "viv_averiadas", "n_movimiento_masa",
             "n_inundacion", "n_vendaval", "n_creciente_subita", "n_avenida_torrencial", "n_otros"]
OBJETIVO = ["tiene_evento", "personas_desplazadas"]


def deduplicar(df):
    df = df.drop_duplicates()
    df = (df.assign(_nulos=df.isna().sum(axis=1)).sort_values(["DIVIPOLA", "fecha", "_nulos"])
            .drop_duplicates(["DIVIPOLA", "fecha"]).drop(columns="_nulos"))
    return df.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)


def preparar(train, test_pub, test_priv):
    partes = {}
    for k, df in {"train": train, "pub": test_pub, "priv": test_priv}.items():
        df = df.copy()
        df["DIVIPOLA"] = df["DIVIPOLA"].astype(str).str.strip().str.zfill(5)
        df["anio"], df["mes"] = df["fecha"].dt.year, df["fecha"].dt.month
        n0 = len(df)
        partes[k] = deduplicar(df)
        if len(partes[k]) != n0:
            print(f"{k}: {n0 - len(partes[k])} filas duplicadas eliminadas")

    tr = partes["train"]
    tr["personas_desplazadas"] = tr["personas_desplazadas"].clip(lower=0)
    tr[HISTORIAL] = tr[HISTORIAL].clip(lower=0).fillna(0)
    viv = tr[["viv_destruidas", "viv_averiadas"]].sum(axis=1)
    afectacion = (tr["personas_desplazadas"].fillna(0) > 0) | (viv > 0)
    corregidas = int(((tr["tiene_evento"] != 1) & afectacion).sum())
    tr["tiene_evento"] = np.where(afectacion, 1, tr["tiene_evento"].fillna(0)).astype(int)
    tr["personas_desplazadas"] = tr["personas_desplazadas"].fillna(0)
    print(f"Objetivo corregido según la definición oficial: {corregidas} filas")

    panel = pd.concat([d.assign(_parte=k, _i=np.arange(len(d))) for k, d in partes.items()], ignore_index=True)
    for c, (lo, hi) in RANGOS.items():
        malo = pd.Series(False, index=panel.index)
        if lo is not None:
            malo |= panel[c] < lo
        if hi is not None:
            malo |= panel[c] > hi
        if malo.any():
            print(f"{c}: {int(malo.sum())} valores imposibles -> nulo")
        panel.loc[malo, c] = np.nan
    faltaba = {c: panel[c].isna() for c in RANGOS if panel[c].isna().any()}

    panel["ONI"] = panel["ONI"].fillna(panel.groupby("fecha")["ONI"].transform("median"))
    for c in COV_ESTATICAS:
        panel[c] = panel[c].fillna(panel.groupby("DIVIPOLA")[c].transform("median"))
    for c in COV_ANUALES:   # el panel está en orden cronológico dentro de cada municipio
        panel[c] = panel[c].fillna(panel.groupby(["DIVIPOLA", "anio"])[c].transform("median"))
        panel[c] = panel.groupby("DIVIPOLA")[c].transform(lambda s: s.ffill().bfill())
    dep = panel["DIVIPOLA"].str[:2]
    for c in COV_ESTATICAS + COV_ANUALES:
        grupos = [dep, panel["anio"]] if c in COV_ANUALES else [dep]
        panel[c] = panel[c].fillna(panel.groupby(grupos)[c].transform("median")).fillna(panel[c].median())
    for c, mascara in faltaba.items():
        panel[c + "_imputado"] = mascara.astype(int)
        print(f"{c}: {int(mascara.sum()):,} valores imputados")
    panel["fase_enso"] = np.select([panel.ONI >= 0.5, panel.ONI <= -0.5], ["El Nino", "La Nina"], "Neutral")

    salida = {k: panel[panel._parte == k].sort_values("_i").drop(columns=["_parte", "_i"]).reset_index(drop=True)
              for k in partes}
    salida["train"]["tiene_evento"] = salida["train"]["tiene_evento"].astype(int)
    for k in ("pub", "priv"):
        salida[k] = salida[k].drop(columns=OBJETIVO + HISTORIAL, errors="ignore")
    return salida["train"], salida["pub"], salida["priv"]


train, test_pub, test_priv = preparar(train, test_pub, test_priv)
print(train.shape, test_pub.shape, test_priv.shape)

Objetivo corregido según la definición oficial: 0 filas
densidad_poblacional: 12 valores imputados
ingresos_tributarios: 36 valores imputados
inversion_gestion_riesgo: 96 valores imputados
nbi_pct: 96 valores imputados
(62814, 29) (20938, 17) (22040, 17)


## 2. Historial, cartografía y variables

In [4]:
K = {"dep": 120, "nm": 1102, "dm": 60, "mun": 24, "mm": 3, "mag": 3}
HIST_COLS = ["hist_tasa_mun", "hist_tasa_mes", "hist_tasa_dep", "hist_tasa_dep_mes", "hist_tasa_nac_mes",
             "hist_mag_mun", "hist_esperado_log", "hist_pct_inund", "hist_pct_masa", "hist_reciente_12"]


def _conteos(hist, claves, p):
    return (hist.groupby(claves).agg(**{f"e_{p}": ("tiene_evento", "sum"), f"n_{p}": ("tiene_evento", "size"),
                                         f"lp_{p}": ("logp", "sum")}).reset_index())


def historial_congelado(hist, filas):
    """Historial calculado con todo `hist` (que es pasado respecto a `filas`), aplicado a `filas`."""
    hist = hist.assign(logp=np.log1p(hist["personas_desplazadas"]), dep=hist["DIVIPOLA"].str[:2])
    f = filas.assign(dep=filas["DIVIPOLA"].str[:2])
    for claves, p in [(["DIVIPOLA"], "mun"), (["DIVIPOLA", "mes"], "mm"), (["dep"], "dep"),
                      (["dep", "mes"], "dm"), (["mes"], "nm")]:
        f = f.merge(_conteos(hist, claves, p), on=claves, how="left")
    c = f.fillna({col: 0 for col in f.columns if col[:2] in ("e_", "n_") or col.startswith("lp_")})
    tasa_nac = hist["tiene_evento"].mean()
    tasa_nm = (c.e_nm + K["nm"] * tasa_nac) / (c.n_nm + K["nm"])
    tasa_dep = (c.e_dep + K["dep"] * tasa_nac) / (c.n_dep + K["dep"])
    tasa_dm = (c.e_dm + K["dm"] * tasa_dep * tasa_nm / tasa_nac) / (c.n_dm + K["dm"])
    tasa_mun = (c.e_mun + K["mun"] * tasa_dep) / (c.n_mun + K["mun"])
    tasa_mm = (c.e_mm + K["mm"] * tasa_mun * tasa_dm / tasa_dep) / (c.n_mm + K["mm"])
    mag_nac = hist["logp"].sum() / max(hist["tiene_evento"].sum(), 1)
    mag_dep = (c.lp_dep + 5 * mag_nac) / (c.e_dep + 5)
    mag_mun = (c.lp_mun + K["mag"] * mag_dep) / (c.e_mun + K["mag"])

    tipos = hist.groupby("DIVIPOLA")[["n_inundacion", "n_movimiento_masa", "n_eventos"]].sum()
    reciente = hist[hist["fecha"] > hist["fecha"].max() - pd.DateOffset(months=12)].groupby("DIVIPOLA")["tiene_evento"].mean()
    salida = filas.copy()
    salida["hist_tasa_mun"] = tasa_mun.to_numpy()
    salida["hist_tasa_mes"] = tasa_mm.to_numpy()
    salida["hist_tasa_dep"] = tasa_dep.to_numpy()
    salida["hist_tasa_dep_mes"] = tasa_dm.to_numpy()
    salida["hist_tasa_nac_mes"] = tasa_nm.to_numpy()
    salida["hist_mag_mun"] = mag_mun.to_numpy()
    salida["hist_esperado_log"] = (tasa_mm * mag_mun).to_numpy()   # E[log1p(personas)] según la historia
    salida["hist_pct_inund"] = filas["DIVIPOLA"].map((tipos.n_inundacion + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_pct_masa"] = filas["DIVIPOLA"].map((tipos.n_movimiento_masa + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_reciente_12"] = filas["DIVIPOLA"].map(reciente).fillna(tasa_nac).to_numpy()
    return salida


def historial_entrenamiento(hist):
    """Filas de entrenamiento: para cada año Y (desde el segundo), historial con los años anteriores a Y."""
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_congelado(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

In [5]:
import io

# Cartografía municipal: centroide y área de cada municipio (MGN del DANE vía la capa generalizada de la UPRA).
GEO = pd.read_csv(io.StringIO("""DIVIPOLA,lat,lon,area_km2
05001,6.2582,-75.6117,369.2
05002,5.8065,-75.4375,501.9
05004,6.6262,-76.0859,289.2
05021,6.3647,-75.0902,132.1
05030,6.0329,-75.7079,81.4
05031,6.978,-74.9815,1210.5
05034,5.6273,-75.9273,406.0
05036,6.1204,-75.716,81.7
05038,6.8669,-75.3567,330.6
05040,7.1937,-75.1085,1407.5
05042,6.53,-75.8994,496.8
05044,6.3085,-75.9123,256.2
05045,7.8964,-76.568,531.4
05051,8.6082,-76.4116,758.4
05055,5.7089,-75.0942,241.2
05059,6.164,-75.8097,110.8
05079,6.4436,-75.3297,206.3
05086,6.6533,-75.6533,297.3
05088,6.3585,-75.589,150.9
05091,5.7309,-75.9853,170.0
05093,6.1841,-75.9558,276.3
05101,5.8406,-76.0093,255.1
05107,7.1111,-75.5629,380.5
05113,6.808,-75.9113,359.8
05120,7.6735,-75.2276,1925.9
05125,6.4237,-75.995,195.3
05129,6.0554,-75.627,133.4
05134,7.0384,-75.2886,202.5
05138,6.8152,-76.0261,360.3
05142,6.3382,-74.7326,263.4
05145,5.5584,-75.6329,79.3
05147,7.789,-76.6536,389.2
05148,5.9733,-75.2554,435.6
05150,6.7585,-75.3162,178.7
05154,7.862,-75.0258,1373.8
05172,7.6313,-76.6395,720.8
05190,6.5501,-75.0832,45.7
05197,6.0187,-75.1652,225.3
05206,6.3845,-75.219,179.9
05209,6.054,-75.9012,238.9
05212,6.3529,-75.5002,69.0
05234,6.9897,-76.3145,2019.0
05237,6.506,-75.3663,199.6
05240,6.3259,-75.7833,238.4
05250,7.7038,-74.6635,1554.5
05264,6.595,-75.5598,215.8
05266,6.1447,-75.57,46.2
05282,5.8868,-75.6806,251.5
05284,6.6849,-76.3344,1344.8
05306,6.6487,-75.9597,124.5
05308,6.3796,-75.4475,80.9
05310,6.7096,-75.1929,330.7
05313,6.1236,-75.1276,189.3
05315,6.8658,-75.2255,121.8
05318,6.2684,-75.4345,164.1
05321,6.2472,-75.1582,84.4
05347,6.2142,-75.7495,113.3
05353,5.7955,-75.907,55.6
05360,6.1794,-75.6161,21.2
05361,7.2595,-75.8979,2277.8
05364,5.571,-75.819,213.4
05368,5.7706,-75.7653,202.5
05376,5.9937,-75.4307,136.2
05380,6.1407,-75.65,37.7
05390,5.7422,-75.5984,65.0
05400,5.941,-75.353,172.0
05411,6.7246,-75.7819,216.7
05425,6.5343,-74.7144,384.0
05440,6.1855,-75.305,114.3
05467,5.9173,-75.5196,75.2
05475,6.8271,-76.7111,1248.6
05480,7.3364,-76.4841,1186.4
05483,5.5819,-75.1938,321.5
05490,8.4934,-76.675,1275.2
05495,7.9903,-74.793,941.0
05501,6.6128,-75.7759,84.9
05541,6.2461,-75.2207,142.4
05543,7.0443,-75.8895,431.3
05576,5.8097,-75.8608,75.0
05579,6.4834,-74.5278,1225.4
05585,6.1397,-74.683,584.0
05591,5.9548,-74.6872,369.4
05604,6.9986,-74.5587,1997.2
05607,6.0693,-75.5134,267.6
05615,6.1517,-75.4102,197.7
05628,6.901,-75.7924,264.7
05631,6.138,-75.6084,15.5
05642,5.9692,-75.9819,297.7
05647,6.9183,-75.6657,218.8
05649,6.1938,-74.9071,716.1
05652,5.8806,-74.976,359.3
05656,6.4382,-75.7049,161.8
05658,6.8232,-75.6783,125.0
05659,8.7076,-76.5319,247.2
05660,6.0203,-74.9381,514.2
05664,6.4589,-75.5628,220.8
05665,8.3708,-76.3193,595.5
05667,6.3103,-74.9977,360.1
05670,6.4485,-74.9322,427.4
05674,6.3118,-75.3242,220.0
05679,5.8861,-75.5848,208.0
05686,6.676,-75.4395,860.4
05690,6.4812,-75.1427,264.8
05697,6.1279,-75.2516,77.6
05736,7.2662,-74.6246,1112.4
05756,5.7622,-75.0614,1357.0
05761,6.5216,-75.746,217.7
05789,5.6742,-75.7101,257.0
05790,7.4848,-75.4891,1713.7
05792,5.8714,-75.829,118.8
05809,6.0581,-75.7996,141.8
05819,7.0168,-75.7132,122.8
05837,7.9892,-76.7284,2992.1
05842,6.9134,-76.1322,264.9
05847,6.336,-76.26,2556.1
05854,7.2646,-75.4002,568.2
05856,5.6509,-75.6312,123.7
05858,6.8368,-74.7543,524.2
05861,5.9563,-75.7772,140.0
05873,6.4522,-76.6936,1644.6
05885,6.7171,-74.7551,442.6
05887,6.9907,-75.4546,714.8
05890,6.6456,-74.9237,1015.1
05893,6.9259,-74.1589,1896.8
05895,7.4751,-74.8579,1175.5
08001,10.9802,-74.8283,144.2
08078,10.7929,-74.922,121.2
08137,10.403,-74.8781,98.6
08141,10.4878,-74.8777,137.0
08296,10.8974,-74.8867,97.1
08372,10.8231,-75.0775,166.9
08421,10.6351,-75.1581,233.9
08433,10.8484,-74.8039,100.0
08436,10.4534,-74.9867,216.7
08520,10.6927,-74.7693,94.5
08549,10.74,-75.1451,252.6
08558,10.7685,-74.854,72.0
08560,10.5935,-74.8009,201.7
08573,11.0014,-74.9095,71.3
08606,10.5092,-75.1208,347.4
08634,10.8,-74.7678,43.3
08638,10.6224,-74.9551,397.7
08675,10.3526,-74.9657,60.4
08685,10.7439,-74.7958,66.4
08758,10.9082,-74.7798,58.3
08770,10.3021,-74.9093,48.2
08832,10.9043,-74.9845,179.7
08849,10.7403,-74.984,103.4
11001,4.3172,-74.1803,1620.2
13001,10.4466,-75.4614,606.9
13006,8.6207,-74.4783,961.8
13030,8.7568,-74.1956,304.3
13042,8.3421,-74.1054,459.8
13052,10.1653,-75.3701,583.9
13062,10.242,-75.0268,164.0
13074,8.8366,-74.1625,428.5
13140,10.2149,-75.0015,255.2
13160,7.2391,-74.0997,876.4
13188,9.2353,-74.6881,133.9
13212,9.5401,-74.8935,584.6
13222,10.556,-75.3141,85.5
13244,9.701,-75.1559,940.4
13248,10.0362,-74.9316,370.6
13268,8.8636,-73.9102,319.0
13300,8.9822,-74.0887,198.6
13430,9.1679,-74.7464,1118.4
13433,10.1782,-75.1794,432.4
13440,9.0519,-74.2242,288.6
13442,9.9819,-75.3441,539.0
13458,8.0186,-74.463,2089.3
13468,9.143,-74.5443,658.7
13473,8.2525,-73.9698,1358.8
13490,8.5057,-74.116,415.7
13549,8.8714,-74.3857,774.8
13580,8.7198,-73.8511,173.1
13600,8.4946,-73.9969,852.5
13620,10.3672,-75.0719,39.7
13647,10.3662,-75.2029,215.6
13650,9.0885,-74.3578,317.9
13654,9.8352,-75.1047,440.0
13655,8.2371,-74.6688,566.8
13657,9.9667,-75.0628,642.8
13667,8.7882,-74.0093,453.5
13670,7.4229,-74.1356,2032.8
13673,10.6436,-75.27,157.6
13683,10.4694,-75.3609,153.4
13688,7.7734,-74.2624,2382.4
13744,7.8515,-73.9836,1357.9
13760,10.3311,-75.1171,92.7
13780,9.3131,-74.64,245.7
13810,8.492,-74.2968,760.4
13836,10.3527,-75.3798,203.8
13838,10.2331,-75.4533,149.2
13873,10.4458,-75.2659,142.9
13894,9.7449,-74.8787,315.8
15001,5.5181,-73.3782,120.1
15022,4.9553,-73.3874,55.3
15047,5.4313,-72.884,880.6
15051,5.7539,-73.4407,143.5
15087,6.0046,-72.8935,161.1
15090,5.2305,-73.1015,57.6
15092,5.9211,-72.8484,101.5
15097,6.3416,-72.621,145.2
15104,5.4397,-73.3823,46.8
15106,5.6752,-73.9084,65.7
15109,5.4917,-73.953,106.1
15114,5.8431,-72.8765,25.0
15131,5.574,-73.8822,81.9
15135,5.0079,-73.1444,300.5
15162,5.9636,-72.9679,70.4
15172,5.2024,-73.3405,137.4
15176,5.6201,-73.8039,173.7
15180,6.6892,-72.4165,658.7
15183,6.0913,-72.4411,680.4
15185,5.9831,-73.427,145.6
15187,5.5642,-73.2634,49.1
15189,5.3925,-73.2818,53.9
15204,5.6924,-73.3334,156.2
15212,5.4448,-74.0264,144.2
15215,5.8248,-72.845,61.6
15218,6.5111,-72.7355,88.4
15223,6.8861,-72.184,1170.6
15224,5.5273,-73.4459,41.3
15226,5.5879,-72.9402,43.6
15232,5.6425,-73.4454,116.1
15236,4.8733,-73.3653,112.5
15238,5.8911,-73.0693,237.4
15244,6.3516,-72.4179,251.0
15248,6.5075,-72.485,74.4
15272,5.6736,-73.0191,110.0
15276,5.8596,-72.925,88.2
15293,5.7444,-73.5423,86.8
15296,5.8031,-72.7344,123.7
15299,5.0892,-73.3141,195.3
15317,6.4456,-72.5169,55.8
15322,5.0195,-73.4856,37.2
15325,4.9344,-73.4923,93.2
15332,6.5684,-72.2545,947.0
15362,5.6175,-72.9704,34.1
15367,5.3776,-73.3784,58.4
15368,6.1457,-72.5858,131.1
15377,5.5409,-72.6009,642.3
15380,5.1128,-73.4583,56.3
15401,5.5229,-74.2248,23.0
15403,6.2648,-72.5563,164.2
15407,5.6657,-73.5149,127.5
15425,4.9798,-73.2942,198.5
15442,5.5701,-74.02,159.8
15455,5.1514,-73.1793,255.9
15464,5.7098,-72.6915,357.7
15466,5.6975,-72.8328,67.2
15469,5.8625,-73.5587,214.8
15476,5.606,-73.3778,59.9
15480,5.5254,-74.1148,136.1
15491,5.779,-72.9333,54.3
15494,5.3548,-73.4487,50.8
15500,5.6107,-73.2802,60.0
15507,5.7551,-74.1999,486.7
15511,5.1484,-73.3995,65.2
15514,5.0999,-73.0033,344.9
15516,5.8245,-73.1405,316.5
15518,5.3852,-72.6982,318.5
15522,6.4438,-72.4621,40.6
15531,5.6898,-74.0046,254.2
15533,5.6234,-72.3702,445.4
15537,6.0288,-72.767,124.4
15542,5.4944,-73.0826,283.7
15550,5.7434,-72.4471,460.0
15572,5.9537,-74.454,1504.4
15580,5.5572,-74.2161,173.2
15599,5.3213,-73.3116,128.3
15600,5.4975,-73.628,217.6
15621,5.3608,-73.2052,159.8
15632,5.6965,-73.7486,232.9
15638,5.5735,-73.5436,63.7
15646,5.4669,-73.5246,176.3
15660,5.2357,-73.0492,110.1
15664,5.9949,-73.5426,71.3
15667,4.8212,-73.1222,469.2
15673,6.4059,-72.5597,123.3
15676,5.5344,-73.7449,86.6
15681,5.6771,-74.1138,197.1
15686,6.0552,-73.4852,69.8
15690,4.8235,-73.2543,311.2
15693,5.8907,-72.9914,114.9
15696,5.7316,-73.6074,86.0
15720,6.1318,-72.7301,160.8
15723,6.0706,-72.7148,51.3
15740,5.4899,-73.2102,146.3
15753,6.3241,-72.6934,117.4
15755,5.933,-72.558,606.7
15757,5.9614,-72.6823,146.8
15759,5.6685,-72.886,209.2
15761,4.9715,-73.4283,58.4
15762,5.5798,-73.4441,46.7
15763,5.7849,-73.2548,284.6
15764,5.4945,-73.3189,56.4
15774,6.2151,-72.7176,190.6
15776,5.6414,-73.6254,101.8
15778,5.0261,-73.4339,39.9
15790,5.8828,-72.7264,212.8
15798,5.0708,-73.4243,46.5
15804,5.3005,-73.393,122.5
15806,5.7479,-73.0112,93.4
15808,5.5788,-73.6751,82.0
15810,6.422,-72.6932,80.0
15814,5.5801,-73.1606,169.1
15816,5.9174,-73.4848,116.4
15820,5.765,-72.836,33.8
15822,5.4896,-73.0048,192.5
15832,5.7181,-73.9453,24.6
15835,5.3022,-73.5116,86.2
15837,5.6726,-73.1831,165.9
15839,6.0773,-72.8565,118.9
15842,5.2172,-73.462,142.5
15861,5.3858,-73.5167,151.7
15879,5.4407,-73.2664,63.0
15897,5.286,-73.167,251.5
17001,5.0836,-75.5078,441.9
17013,5.5931,-75.4507,474.6
17042,5.1962,-75.754,211.8
17050,5.2657,-75.4743,148.9
17088,4.9995,-75.8116,105.7
17174,4.9937,-75.6717,107.0
17272,5.2868,-75.5914,195.2
17380,5.5382,-74.7114,541.1
17388,5.3941,-75.5578,89.4
17433,5.2351,-75.153,194.8
17442,5.4914,-75.6018,35.9
17444,5.308,-75.0412,93.9
17446,5.2322,-75.2832,388.2
17486,5.1848,-75.5092,376.1
17495,5.638,-74.8506,233.1
17513,5.5004,-75.483,260.0
17524,5.0505,-75.6624,112.0
17541,5.4065,-75.1657,491.8
17614,5.441,-75.7495,379.0
17616,5.1126,-75.7594,89.5
17653,5.3701,-75.4093,390.0
17662,5.5409,-74.994,757.5
17665,5.0709,-75.813,60.7
17777,5.4561,-75.6421,125.3
17867,5.449,-74.8368,552.1
17873,4.9384,-75.4371,455.7
17877,5.0447,-75.8779,103.4
18001,1.749,-75.5618,2584.7
18029,1.228,-75.8818,410.6
18094,1.494,-75.8775,1141.9
18150,0.6453,-74.2725,12701.3
18205,1.0806,-75.9654,494.6
18247,1.7935,-75.1957,1105.1
18256,1.6137,-75.2308,1246.2
18410,1.3032,-75.2356,1710.3
18460,1.1433,-75.3832,1223.5
18479,1.3785,-75.6731,476.6
18592,2.0184,-75.0726,4149.0
18610,1.315,-76.1132,1239.8
18753,1.5648,-74.2609,17555.3
18756,0.2116,-73.1273,42345.1
18785,0.9331,-75.6683,698.7
18860,1.0873,-75.6161,1032.4
19001,2.4718,-76.5917,481.7
19022,1.9124,-76.8345,234.3
19050,2.329,-77.2632,774.6
19075,2.0841,-77.2081,417.4
19100,1.8545,-76.9635,800.0
19110,2.9928,-76.6243,233.6
19130,2.6432,-76.7018,553.3
19137,2.8102,-76.4656,353.1
19142,3.0453,-76.366,266.5
19212,3.1362,-76.2108,323.6
19256,2.5352,-77.0053,2737.8
19290,1.7001,-77.0807,52.4
19300,3.1438,-76.392,98.3
19318,2.4132,-77.6895,2552.0
19355,2.4987,-76.1373,873.5
19364,2.85,-76.3125,232.3
19392,2.1876,-76.7815,208.6
19397,2.0545,-76.7605,510.8
19418,3.0016,-77.2199,3359.5
19450,1.7939,-77.18,690.2
19455,3.2263,-76.2111,192.2
19473,2.8147,-76.7511,510.0
19513,3.1955,-76.3362,69.6
19517,2.7346,-75.9948,1790.9
19532,2.1481,-77.0576,755.1
19533,1.1516,-76.3076,1092.8
19548,2.7139,-76.5648,180.8
19573,3.2659,-76.4157,108.0
19585,2.2838,-76.3728,846.6
19622,2.2516,-76.7538,166.9
19693,1.8576,-76.7163,422.6
19698,2.985,-76.4989,518.0
19701,1.5201,-76.5388,3620.3
19743,2.6521,-76.3503,678.0
19760,2.2297,-76.6112,520.2
19780,2.9946,-76.7445,594.0
19785,2.0579,-76.9092,134.7
19807,2.3693,-76.7102,199.4
19809,2.6996,-77.4918,2051.5
19821,2.9765,-76.2031,489.3
19824,2.5047,-76.3907,403.1
19845,3.1818,-76.4642,84.2
20001,10.3811,-73.4634,4161.9
20011,8.2476,-73.6243,871.6
20013,9.9488,-73.2238,1756.2
20032,9.5105,-73.9399,640.7
20045,9.7285,-73.2499,1231.3
20060,9.9459,-73.8651,591.6
20175,9.2202,-73.7528,1370.2
20178,9.4273,-73.547,1119.6
20228,9.2381,-73.4874,914.0
20238,10.1994,-73.9123,952.0
20250,9.6828,-73.6934,809.6
20295,8.339,-73.7071,328.8
20310,8.3967,-73.3785,60.2
20383,8.6211,-73.6315,793.6
20400,9.5521,-73.3201,752.1
20443,10.39,-73.0002,132.4
20517,8.9389,-73.5717,532.8
20550,8.7651,-73.6315,426.0
20570,10.4662,-73.588,743.9
20614,8.2016,-73.4863,559.9
20621,10.1649,-73.189,1076.8
20710,7.8132,-73.457,542.4
20750,10.1699,-73.2879,643.5
20770,7.9378,-73.5447,991.7
20787,8.9017,-73.7614,512.9
23001,8.5839,-75.9501,3142.4
23068,8.2605,-75.0491,1927.0
23079,8.1921,-75.4336,839.1
23090,8.723,-76.2376,419.7
23162,8.891,-75.834,289.0
23168,9.1104,-75.6519,326.3
23182,9.03,-75.3643,583.2
23189,8.8421,-75.6078,643.4
23300,9.0572,-75.7783,87.8
23350,8.0526,-75.2878,288.6
23417,9.1647,-75.9102,947.7
23419,8.8379,-76.2946,368.5
23464,9.2659,-75.648,167.5
23466,7.8844,-75.7627,1545.7
23500,9.1911,-76.1193,204.9
23555,8.2901,-75.6132,1141.8
23570,8.4827,-75.4131,809.1
23574,8.9892,-76.1933,411.4
23580,7.7127,-75.77,1638.8
23586,9.288,-75.7184,124.7
23660,8.7992,-75.4289,965.9
23670,9.1235,-75.5161,218.1
23672,9.3575,-75.7893,202.9
23675,9.3316,-75.972,318.0
23678,8.7062,-75.6922,444.1
23682,7.7545,-75.566,514.1
23686,8.9877,-75.9091,444.2
23807,7.8126,-76.1688,4915.0
23815,9.226,-75.5452,107.6
23855,8.2208,-76.2192,914.8
25001,4.3722,-74.6707,85.0
25019,4.8918,-74.4428,51.7
25035,4.5602,-74.5254,121.5
25040,4.7891,-74.4614,120.3
25053,4.2324,-74.3949,143.7
25086,4.7215,-74.7544,180.6
25095,4.8481,-74.5288,62.9
25099,4.692,-74.3294,104.5
25120,3.9185,-74.4444,418.3
25123,4.7224,-74.4608,51.2
25126,4.9298,-74.0294,50.1
25148,5.382,-74.5213,613.8
25151,4.4024,-73.9441,112.0
25154,5.3401,-73.9252,299.6
25168,4.955,-74.6332,174.9
25175,4.8651,-74.0372,80.0
25178,4.4265,-74.0587,148.4
25181,4.5819,-73.9128,211.8
25183,5.117,-73.6813,301.9
25200,5.1035,-73.9749,133.2
25214,4.787,-74.1235,55.4
25224,5.2301,-73.7792,110.9
25245,4.5569,-74.4267,117.3
25258,5.2611,-74.3089,134.4
25260,4.8795,-74.238,85.6
25269,4.8342,-74.3378,159.4
25279,4.5263,-73.7966,461.3
25281,4.3181,-73.9534,112.1
25286,4.7434,-74.1988,67.9
25288,5.415,-73.7714,76.4
25290,4.3191,-74.3846,209.6
25293,4.6527,-73.5126,399.5
25295,5.011,-73.8766,42.8
25297,4.8677,-73.6151,269.6
25299,4.7236,-73.601,105.2
25307,4.3384,-74.8064,127.7
25312,4.5231,-74.3374,60.3
25317,5.3884,-73.6951,171.0
25320,5.1755,-74.6405,767.5
25322,4.8017,-73.8463,357.6
25324,4.5096,-74.7828,94.0
25326,4.9123,-73.7862,253.5
25328,4.8789,-74.4831,61.4
25335,4.2311,-73.8388,220.6
25339,4.1684,-74.0244,451.4
25368,4.5707,-74.6896,220.5
25372,4.7041,-73.6932,346.4
25377,4.7018,-73.9288,328.4
25386,4.6506,-74.4738,148.6
25394,5.3323,-74.4085,190.0
25398,5.2045,-74.4082,129.4
25402,4.9783,-74.3392,155.4
25407,5.3031,-73.6823,153.0
25426,5.0398,-73.6171,230.5
25430,4.7729,-74.2596,120.0
25436,4.9753,-73.5629,107.0
25438,4.4861,-73.4053,1182.5
25473,4.673,-74.2357,104.0
25483,4.404,-74.8077,56.4
25486,5.0932,-73.8839,97.2
25488,4.2949,-74.6144,227.2
25489,5.1328,-74.3968,57.1
25491,5.0662,-74.3901,68.8
25506,4.0655,-74.4502,123.3
25513,5.1685,-74.1635,405.9
25518,5.3872,-74.1652,172.9
25524,4.1804,-74.4727,74.4
25530,4.4518,-73.1892,903.0
25535,4.2948,-74.2702,256.8
25572,5.6196,-74.5794,510.2
25580,4.6918,-74.6804,197.3
25592,5.1001,-74.494,79.4
25594,4.3345,-73.8523,137.9
25596,4.7145,-74.5546,129.2
25599,4.5233,-74.5773,117.2
25612,4.3076,-74.7292,124.6
25645,4.599,-74.3449,89.5
25649,4.1412,-74.3535,244.2
25653,5.317,-74.0715,298.0
25658,4.9643,-74.2717,117.7
25662,4.8398,-74.6565,308.3
25718,4.947,-74.4124,111.0
25736,5.0091,-73.7818,136.3
25740,4.4658,-74.2629,122.2
25743,4.4366,-74.3706,162.9
25745,5.5046,-73.8467,90.2
25754,4.5375,-74.2303,186.4
25758,4.888,-73.9657,110.7
25769,4.9693,-74.1573,211.1
25772,5.1358,-73.7826,173.5
25777,5.0617,-74.2317,130.0
25779,5.4435,-73.8192,109.9
25781,5.2432,-73.8552,64.1
25785,4.9517,-74.0838,76.2
25793,5.1941,-73.9558,203.6
25797,4.6519,-74.3919,51.9
25799,4.8386,-74.1468,113.6
25805,4.3205,-74.4887,86.4
25807,5.0823,-73.5148,57.3
25815,4.4518,-74.6478,253.2
25817,4.9682,-73.9284,74.3
25823,5.3554,-74.2892,142.1
25839,4.8016,-73.4782,512.3
25841,4.4997,-73.9742,108.6
25843,5.3159,-73.8196,102.6
25845,4.3006,-74.0701,208.8
25851,5.1925,-74.4786,91.2
25862,5.1306,-74.3104,145.2
25867,4.8894,-74.5537,65.5
25871,5.2704,-74.1906,62.7
25873,5.2285,-73.5781,229.6
25875,5.0114,-74.488,141.1
25878,4.4361,-74.4861,201.8
25885,5.5922,-74.3656,961.2
25898,4.7538,-74.3872,52.9
25899,5.0454,-74.023,195.2
27001,5.9389,-76.663,3517.7
27006,8.4346,-77.2885,810.5
27025,5.6379,-77.0842,2063.1
27050,5.5598,-76.6053,432.4
27073,5.5048,-76.1974,811.0
27075,6.2952,-77.3553,913.0
27077,5.0112,-77.2159,3438.1
27099,6.4755,-77.0997,3618.8
27135,5.36,-76.7726,378.7
27150,7.0337,-76.9749,3172.0
27160,5.3891,-76.5226,419.7
27205,5.0738,-76.5272,467.7
27245,5.8091,-76.193,818.5
27250,4.247,-76.9607,4094.5
27361,4.738,-76.8688,1876.2
27372,7.1113,-77.6834,1313.8
27413,5.5852,-76.3833,830.8
27425,6.0343,-76.6734,1808.5
27430,5.1106,-76.9954,1370.6
27450,4.8121,-76.8029,676.8
27491,4.8525,-76.6181,941.2
27495,5.6898,-77.2936,704.5
27580,5.1732,-76.4839,325.8
27600,5.5664,-76.8108,688.8
27615,7.3894,-77.2517,7427.0
27660,4.9582,-76.2811,1578.7
27745,4.5926,-76.541,1554.8
27787,5.2668,-76.3309,713.7
27800,8.1193,-77.1052,1172.4
27810,5.2848,-76.6454,177.8
41001,3.0067,-75.2812,1203.4
41006,1.71,-75.9962,539.7
41013,2.2685,-75.7164,266.6
41016,3.2626,-75.3501,794.4
41020,2.4974,-75.2923,584.0
41026,2.079,-75.769,178.8
41078,3.1342,-74.9576,795.2
41132,2.6587,-75.329,461.1
41206,3.454,-74.6776,1592.3
41244,2.006,-75.9518,79.6
41298,2.1739,-75.5709,611.9
41306,2.3717,-75.517,503.6
41319,1.9868,-75.6966,250.0
41349,2.5413,-75.4474,196.5
41357,2.6921,-75.6741,358.9
41359,2.0061,-76.2558,373.3
41378,2.1797,-76.1138,343.7
41396,2.3251,-76.0025,822.6
41483,2.5746,-75.7871,123.8
41503,2.0613,-76.0413,155.4
41518,2.4061,-75.7284,277.1
41524,2.9148,-75.4412,884.2
41530,1.6877,-76.1472,173.3
41548,2.2604,-75.8319,192.1
41551,1.8253,-76.1208,633.7
41615,2.7778,-75.2083,326.1
41660,2.1,-76.2283,465.5
41668,1.9238,-76.4154,1378.3
41676,2.9266,-75.6323,335.4
41770,1.8714,-75.807,431.8
41791,2.1308,-75.8756,365.4
41797,2.5265,-75.6797,386.8
41799,3.0315,-75.0857,521.6
41801,2.8228,-75.7109,469.5
41807,1.9518,-75.9168,187.8
41872,3.2845,-75.1374,530.2
41885,2.6512,-75.5141,320.9
44001,11.2429,-72.9587,3081.7
44035,11.2285,-72.5318,548.9
44078,10.9889,-72.6882,798.6
44090,11.0895,-73.4352,1761.8
44098,10.9259,-72.9445,232.7
44110,10.6354,-72.884,245.6
44279,10.8289,-72.802,469.3
44378,11.0999,-72.7329,214.8
44420,10.46,-73.076,171.3
44430,11.3815,-72.2952,1764.9
44560,11.6059,-72.5708,1625.3
44650,10.8126,-73.0878,1319.6
44847,11.9996,-71.7495,7866.9
44855,10.4953,-72.9886,241.9
44874,10.5836,-72.9746,261.5
47001,11.1227,-73.8861,2344.5
47030,10.2525,-74.1022,409.1
47053,10.6453,-73.8787,1748.4
47058,9.8342,-74.0851,1125.9
47161,10.2934,-74.8184,178.7
47170,10.0995,-74.5267,539.0
47189,10.8899,-74.0433,1329.2
47205,10.2326,-74.7781,105.7
47245,9.1179,-73.9804,840.0
47258,10.3433,-74.6787,556.7
47268,10.6594,-74.3258,264.6
47288,10.4587,-73.8873,965.9
47318,9.2458,-74.1427,544.4
47460,9.7371,-74.3289,848.3
47541,10.1529,-74.8287,316.7
47545,9.5262,-74.1505,673.0
47551,10.433,-74.4131,1640.5
47555,9.7683,-74.6024,1458.0
47570,10.849,-74.36,683.0
47605,10.649,-74.5819,599.0
47660,10.1058,-74.2639,1239.4
47675,10.5162,-74.7218,167.6
47692,9.3525,-74.202,446.2
47703,9.3256,-74.358,253.3
47707,9.5194,-74.3389,1124.8
47720,9.5219,-74.6476,495.9
47745,10.8904,-74.6275,969.7
47798,9.9263,-74.7308,489.9
47960,10.1185,-74.6913,350.7
47980,10.7959,-74.1781,445.7
50001,4.0923,-73.4891,1323.2
50006,4.009,-73.7234,1117.2
50110,4.5193,-72.9935,412.6
50124,4.3126,-72.9518,935.8
50150,3.8053,-73.5381,508.7
50223,3.8276,-74.0741,1147.7
50226,4.2318,-73.3164,624.6
50245,4.3513,-73.7111,262.0
50251,3.6155,-73.8872,574.1
50270,3.7077,-73.8316,116.5
50287,3.3831,-73.5959,572.7
50313,3.5013,-73.7449,335.6
50318,3.9475,-73.9597,605.0
50325,3.1174,-71.9407,11911.1
50330,3.1524,-74.1245,2292.2
50350,2.162,-74.094,10853.3
50370,3.0488,-74.4315,6422.8
50400,3.6147,-74.0962,815.1
50450,2.7497,-72.7227,1257.2
50568,4.0054,-71.6317,17271.0
50573,4.0131,-72.6429,6848.5
50577,3.1905,-73.2354,2525.1
50590,2.7551,-73.1381,3383.6
50606,4.2404,-73.4758,374.5
50680,3.8467,-73.2761,802.6
50683,3.2916,-73.8167,1185.1
50686,4.4666,-73.6664,228.5
50689,3.5352,-72.985,5952.4
50711,2.8131,-73.6666,4834.1
52001,1.0791,-77.2025,1081.8
52019,1.4694,-77.0692,39.2
52022,0.9118,-77.6952,46.4
52036,1.2453,-77.5308,67.9
52051,1.4792,-77.1293,59.4
52079,1.445,-78.1566,2726.2
52083,1.5908,-77.0424,42.7
52110,1.3157,-77.1174,628.8
52203,1.6369,-77.0482,59.6
52207,1.2092,-77.4408,119.7
52210,0.9324,-77.528,40.8
52215,0.7673,-77.3556,312.4
52224,0.8739,-77.7342,59.2
52227,0.9449,-77.9602,913.5
52233,1.725,-77.5921,353.7
52240,1.3872,-77.2697,145.7
52250,2.1868,-77.7988,2473.5
52254,1.5122,-77.4294,122.7
52256,1.8458,-77.44,515.6
52258,1.4128,-76.9837,308.7
52260,1.4297,-77.3822,246.0
52287,0.9138,-77.3271,399.0
52317,0.9749,-77.7378,155.0
52320,1.1515,-77.5303,116.6
52323,0.9295,-77.5824,30.0
52352,0.9806,-77.5191,82.0
52354,1.0735,-77.5016,80.7
52356,0.5624,-77.3683,1532.7
52378,1.585,-76.9268,228.2
52381,1.3334,-77.388,136.3
52385,1.5541,-77.7044,240.6
52390,2.4238,-78.2092,420.4
52399,1.6206,-77.1389,144.5
52405,1.9389,-77.3137,309.5
52411,1.395,-77.5204,135.8
52418,1.6721,-77.7104,955.1
52427,1.9073,-78.0448,1806.5
52435,1.1553,-77.8467,563.8
52473,2.4424,-78.439,757.8
52480,1.3038,-77.3455,48.8
52490,2.2898,-78.2945,995.1
52506,1.0303,-77.5522,64.3
52520,2.0903,-78.5907,513.5
52540,1.7358,-77.4814,431.1
52560,0.7252,-77.4274,373.8
52565,1.2323,-77.5979,39.8
52573,0.8264,-77.3223,351.4
52585,0.917,-77.6334,130.0
52612,1.2035,-78.0465,1045.6
52621,1.8979,-78.3808,1455.5
52678,1.4302,-77.6914,603.3
52683,1.2889,-77.4574,99.1
52685,1.53,-77.0205,66.0
52687,1.5432,-77.2181,251.1
52693,1.6789,-76.9729,105.6
52694,1.537,-77.1015,54.3
52696,2.2942,-77.869,1216.0
52699,1.2845,-77.7443,535.8
52720,1.0364,-77.6804,115.1
52786,1.5944,-77.3244,240.7
52788,1.065,-77.3507,215.2
52835,1.57,-78.6133,3588.9
52838,1.1344,-77.6308,215.4
52885,1.125,-77.4248,101.6
54001,8.1123,-72.4886,1144.0
54003,8.0207,-73.1587,1382.7
54051,7.6145,-72.8595,456.7
54099,7.6316,-72.6601,175.9
54109,8.0806,-72.9298,268.4
54125,7.2705,-72.6524,138.6
54128,7.6856,-73.1273,621.9
54172,7.5829,-72.5838,169.2
54174,7.0522,-72.5252,1177.4
54206,8.8614,-73.2061,943.6
54223,7.5019,-72.7968,365.4
54239,7.7475,-72.6718,175.7
54245,8.8664,-73.3456,1721.2
54250,8.6775,-73.0319,705.6
54261,8.0809,-72.6331,516.1
54313,7.9074,-72.8063,151.2
54344,8.3624,-73.078,419.8
54347,7.4902,-72.489,115.3
54377,7.2643,-72.5226,254.7
54385,7.6961,-73.362,676.9
54398,8.2569,-73.1885,240.4
54405,7.7398,-72.5277,125.1
54418,7.964,-72.8475,87.9
54480,7.3175,-72.7712,167.0
54498,8.2079,-73.3365,513.0
54518,7.3791,-72.6757,297.2
54520,7.4829,-72.6348,171.0
54553,8.3282,-72.4119,42.5
54599,7.5976,-72.5083,94.2
54660,7.7738,-72.8683,492.2
54670,8.4545,-73.1588,396.4
54673,7.8491,-72.6102,138.7
54680,7.8892,-72.7186,170.2
54720,8.2658,-72.807,1461.7
54743,7.1792,-72.7838,312.1
54800,8.762,-73.1562,928.0
54810,8.7128,-72.8002,2649.2
54820,7.2257,-72.3069,1473.6
54871,7.9364,-72.988,397.5
54874,7.7536,-72.4826,89.8
63001,4.4976,-75.7261,119.5
63111,4.3677,-75.7445,35.1
63130,4.466,-75.6592,229.1
63190,4.6046,-75.6663,91.6
63212,4.3964,-75.6496,95.2
63272,4.6688,-75.6762,112.6
63302,4.1795,-75.7592,293.5
63401,4.4362,-75.8143,86.1
63470,4.521,-75.8189,152.7
63548,4.3032,-75.6839,251.1
63594,4.6089,-75.7885,126.6
63690,4.6121,-75.5295,347.8
66001,4.7823,-75.72,604.5
66045,5.1364,-75.9496,156.8
66075,4.9204,-75.9406,111.4
66088,5.2061,-75.8673,179.5
66170,4.8416,-75.6703,70.2
66318,5.3209,-75.8051,104.6
66383,4.9866,-76.0084,96.8
66400,4.9116,-75.8444,42.0
66440,4.9383,-75.7536,139.1
66456,5.4127,-75.9611,564.5
66572,5.2847,-76.0727,613.1
66594,5.3256,-75.7172,120.8
66682,4.8349,-75.5509,557.1
66687,5.0457,-75.9827,195.3
68001,7.1564,-73.1113,153.5
68013,6.1825,-73.5299,74.8
68020,5.7777,-73.8314,168.9
68051,6.7172,-73.0109,168.4
68077,5.9578,-73.6219,47.3
68079,6.6478,-73.2151,135.4
68081,7.0548,-73.7776,1341.8
68092,7.0133,-73.3873,429.6
68101,6.0953,-74.1096,995.2
68121,6.5628,-73.2498,67.9
68132,7.3527,-72.9266,44.0
68147,6.525,-72.6775,83.4
68152,6.6595,-72.5709,253.6
68160,6.7517,-72.9283,108.4
68162,6.9019,-72.6474,410.0
68167,6.1992,-73.1556,413.1
68169,7.2592,-72.9881,127.1
68176,6.3714,-73.414,170.1
68179,6.0682,-73.6314,97.6
68190,6.41,-74.1706,3168.8
68207,6.7856,-72.6045,342.5
68209,6.3552,-73.2401,70.5
68211,6.3028,-73.5009,136.4
68217,6.234,-72.982,584.1
68229,6.619,-73.0192,244.4
68235,6.6644,-73.5651,916.3
68245,6.2537,-73.5365,93.4
68250,6.0964,-73.9263,397.2
68255,7.5244,-73.1805,456.3
68264,6.0801,-73.062,397.4
68266,6.658,-72.6795,73.7
68271,5.8026,-73.9545,171.2
68276,7.0792,-73.0681,99.6
68296,6.6708,-73.3249,203.4
68298,5.9324,-73.3168,577.0
68307,7.0372,-73.2495,498.0
68318,6.9436,-72.8239,309.3
68320,6.2336,-73.4097,146.1
68322,6.3199,-73.3315,65.6
68324,5.9444,-73.7215,82.1
68327,6.0326,-73.5801,29.0
68344,6.5606,-73.3631,166.8
68368,5.8536,-73.8247,89.4
68370,6.7138,-73.0999,39.8
68377,5.9096,-74.0534,345.6
68385,6.3311,-73.8187,597.4
68397,6.2164,-73.6286,282.7
68406,7.205,-73.2917,552.1
68418,6.8121,-73.1077,290.6
68425,6.5188,-72.5822,102.2
68432,6.7202,-72.7381,57.1
68444,7.3486,-73.0549,232.9
68464,6.4925,-72.958,487.1
68468,6.639,-72.8176,188.0
68498,6.3572,-73.1217,75.7
68500,6.2265,-73.2799,281.7
68502,6.3347,-72.8129,480.2
68522,6.5256,-73.29,20.6
68524,6.3923,-73.2826,59.8
68533,6.4167,-73.1715,74.2
68547,6.9619,-73.0058,479.3
68549,6.5137,-73.1755,54.1
68572,5.8314,-73.6819,259.5
68573,6.685,-73.9574,760.5
68575,7.5701,-73.7842,1485.5
68615,7.4746,-73.373,1169.0
68655,7.407,-73.5717,1398.9
68669,6.7969,-72.8151,280.5
68673,6.1028,-73.5272,55.7
68679,6.5498,-73.1185,147.6
68682,6.4606,-72.8539,140.9
68684,6.6266,-72.74,80.2
68686,6.5681,-72.6428,71.7
68689,6.891,-73.5372,1101.1
68705,6.9894,-72.896,223.6
68720,6.4087,-73.5814,375.1
68745,6.6478,-73.6896,912.2
68755,6.4627,-73.2444,127.7
68770,6.1147,-73.369,292.6
68773,5.9831,-73.9591,523.4
68780,7.473,-72.987,369.8
68820,7.1721,-72.9392,341.0
68855,6.4354,-73.1097,79.4
68861,6.2204,-73.7024,448.6
68867,7.3218,-72.8852,85.4
68872,6.6858,-73.1635,99.1
68895,6.8334,-73.2788,363.4
70001,9.3118,-75.4313,284.0
70110,9.3287,-74.951,116.1
70124,8.7833,-75.1302,423.0
70204,9.5269,-75.362,130.3
70215,9.2032,-75.2483,274.4
70221,9.404,-75.6474,47.1
70230,9.57,-75.3202,85.8
70233,9.0686,-75.1382,201.4
70235,9.1359,-74.9801,312.0
70265,8.3801,-74.6812,339.9
70400,8.8126,-75.2846,238.9
70418,9.4251,-75.2139,204.1
70429,8.5482,-74.7089,883.0
70473,9.3918,-75.3277,180.3
70508,9.5494,-75.1853,438.3
70523,9.3408,-75.5638,156.8
70670,9.162,-75.3492,218.8
70678,8.8242,-74.9677,1606.8
70702,9.29,-75.214,164.1
70708,8.5815,-75.1823,963.8
70713,9.8113,-75.5013,1081.2
70717,9.4009,-75.0348,227.3
70742,9.2559,-75.0872,453.7
70771,8.8195,-74.7467,1093.0
70820,9.5332,-75.5367,319.1
70823,9.4904,-75.4426,285.0
73001,4.4515,-75.2518,1377.4
73024,3.3919,-74.9405,493.4
73026,4.582,-74.9873,338.2
73030,4.8226,-74.8054,236.0
73043,4.6324,-75.1934,472.8
73055,5.0064,-74.8452,445.5
73067,3.4284,-75.482,1018.7
73124,4.4044,-75.4981,506.6
73148,4.1238,-74.746,189.3
73152,5.0181,-75.1858,179.0
73168,3.7448,-75.5902,2103.5
73200,4.3538,-74.9122,338.3
73217,3.7368,-75.1482,665.0
73226,3.9832,-74.6891,507.1
73236,3.5962,-74.8041,652.1
73268,4.1672,-74.8932,220.6
73270,5.0794,-74.9569,183.3
73275,4.2446,-74.8418,97.2
73283,5.1851,-75.0538,219.7
73319,4.076,-74.9764,505.8
73347,5.0686,-75.2411,313.5
73349,5.1801,-74.7841,299.5
73352,4.1281,-74.5403,215.0
73408,4.8658,-74.9227,271.0
73411,4.8776,-75.0477,284.9
73443,5.2364,-74.9062,296.8
73449,4.1844,-74.6393,196.3
73461,4.8278,-75.2209,423.1
73483,3.5393,-75.1216,866.7
73504,3.9366,-75.2768,950.0
73520,5.0949,-75.018,64.9
73547,4.4853,-74.9206,352.3
73555,3.0993,-75.8172,1753.4
73563,3.7282,-74.8711,415.8
73585,3.8546,-74.8749,406.6
73616,3.4678,-75.8548,2054.1
73622,4.0976,-75.594,763.2
73624,4.2164,-75.3472,742.3
73671,3.9112,-75.0212,197.1
73675,3.9568,-75.4971,391.0
73678,4.1286,-75.1123,409.4
73686,4.7292,-75.2069,272.1
73770,4.0492,-74.8183,190.9
73854,4.1827,-75.1712,196.7
73861,4.7085,-74.921,342.1
73870,4.9644,-75.1565,278.4
73873,3.854,-74.6196,429.0
76001,3.3981,-76.5748,564.1
76020,4.6812,-75.7904,54.1
76036,4.145,-76.1646,110.4
76041,4.7972,-76.0312,309.7
76054,4.7048,-76.1419,89.9
76100,4.3652,-76.3631,736.3
76109,3.6178,-77.0884,6355.9
76111,3.8558,-76.1066,822.7
76113,4.1971,-76.0899,394.4
76122,4.3061,-75.8382,160.9
76126,3.9434,-76.629,791.8
76130,3.3823,-76.3844,291.2
76147,4.7038,-75.916,253.7
76233,3.6496,-76.7207,917.2
76243,4.9121,-76.0694,186.2
76246,4.756,-76.2131,202.2
76248,3.6764,-76.208,443.0
76250,4.5318,-76.2833,201.2
76275,3.3056,-76.1674,400.5
76306,3.7445,-76.1899,263.5
76318,3.7835,-76.2947,163.9
76364,3.2151,-76.6294,619.7
76377,3.6938,-76.59,257.7
76400,4.5384,-76.1014,119.6
76403,4.4834,-75.9678,258.5
76497,4.5976,-75.9488,214.9
76520,3.5584,-76.226,996.0
76563,3.4209,-76.1781,357.5
76606,3.8058,-76.5345,135.2
76616,4.1094,-76.3734,311.0
76622,4.444,-76.1685,236.3
76670,3.9821,-76.2072,211.4
76736,4.1939,-75.8894,533.9
76823,4.6382,-76.081,178.6
76828,4.2327,-76.3463,315.3
76834,4.0283,-76.0317,901.5
76845,4.7098,-75.7894,39.4
76863,4.6211,-76.232,240.7
76869,3.7433,-76.4704,112.3
76890,3.904,-76.3918,324.8
76892,3.5952,-76.509,238.0
76895,4.3543,-76.0488,376.4
81001,6.7962,-70.5094,5778.2
81065,6.7972,-71.2985,3069.0
81220,6.3481,-70.0708,5153.3
81300,6.7044,-71.8309,1154.6
81591,6.411,-70.9686,2280.2
81736,6.9058,-71.8507,939.1
81794,6.3904,-71.7936,5359.8
85001,5.2422,-72.2575,2485.2
85010,5.1272,-72.5484,1455.2
85015,5.2246,-72.8637,295.4
85125,6.1098,-71.1339,5479.7
85136,6.1767,-72.3466,192.7
85139,4.6757,-72.2141,3733.2
85162,4.8431,-72.8481,783.4
85225,5.529,-72.0918,1102.0
85230,4.9152,-71.5994,4719.2
85250,5.7806,-70.8702,12097.9
85263,5.6627,-71.9251,781.6
85279,5.2602,-72.77,174.4
85300,4.8073,-72.9962,398.4
85315,6.0523,-72.21,313.7
85325,5.2314,-71.5925,2994.5
85400,5.8795,-72.199,1094.2
85410,4.6978,-72.6289,2375.6
85430,5.3677,-71.1965,2935.4
85440,4.5284,-72.8214,832.2
86001,1.2216,-76.68,1325.2
86219,1.2223,-76.9698,72.4
86320,0.6741,-76.9424,1925.3
86568,0.4784,-76.3291,2811.9
86569,0.7248,-76.4813,926.9
86571,0.7444,-75.8767,4544.0
86573,0.0611,-75.0649,10770.1
86749,1.2336,-76.9136,104.9
86755,1.1363,-76.8477,406.0
86757,0.3056,-76.8562,380.7
86760,1.0813,-77.0023,331.4
86865,0.4142,-76.8833,812.8
86885,0.9112,-76.7434,1390.3
88001,12.5434,-81.7171,26.0
88564,13.3509,-81.3738,22.2
91001,-3.5286,-70.0425,6173.2
91263,-1.9931,-72.7251,10675.3
91405,-1.2439,-72.7573,12800.9
91407,-1.4282,-70.038,13632.5
91430,-0.1266,-71.1046,1435.2
91460,-0.6884,-71.1844,16856.6
91530,-0.9512,-73.7535,8376.0
91536,-1.9094,-71.149,13518.1
91540,-3.635,-70.4707,1460.7
91669,-1.103,-71.9366,14747.8
91798,-2.4492,-70.0026,9104.1
94001,3.3181,-68.4579,15814.6
94343,3.3131,-69.3988,9414.5
94663,2.8094,-70.2681,4911.2
94883,2.1466,-67.3372,3001.0
94884,2.508,-68.2076,15547.9
94885,1.4082,-67.0047,1188.6
94886,3.3852,-67.5851,2292.0
94887,1.981,-69.1148,10081.0
94888,2.4207,-69.8386,8515.1
95001,2.4843,-71.9154,16811.3
95015,1.6144,-73.0341,13559.2
95025,2.0851,-71.5345,12355.1
95200,1.3636,-72.0181,12786.6
97001,1.0159,-70.4153,16202.0
97161,1.0534,-71.3311,6365.0
97511,0.1932,-70.9312,14016.6
97666,-0.7159,-69.9064,6454.0
97777,1.6815,-70.706,5527.0
97889,0.8329,-69.6184,4633.2
99001,5.8351,-68.1416,12089.1
99524,5.5213,-69.6106,18284.9
99624,4.9802,-70.6512,3904.5
99773,4.262,-69.5218,65097.4"""), dtype={"DIVIPOLA": str})


def vecinos_mas_cercanos(geo, k=5):
    """Los k municipios con el centroide más cercano (distancia sobre la esfera)."""
    lat, lon = np.radians(geo["lat"].to_numpy()), np.radians(geo["lon"].to_numpy())
    a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
    distancia = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    np.fill_diagonal(distancia, np.inf)
    cercanos = np.argsort(distancia, axis=1)[:, :k]
    codigos = geo["DIVIPOLA"].to_numpy()
    return pd.DataFrame({"DIVIPOLA": np.repeat(codigos, k), "vecino": codigos[cercanos.ravel()]})


VECINOS = vecinos_mas_cercanos(GEO)
MESES = pd.DataFrame({"mes": range(1, 13)})
faltan = sorted(set(train["DIVIPOLA"]) - set(GEO["DIVIPOLA"]))
print(f"Municipios del reto sin cartografía: {len(faltan)}", faltan[:10])
train, test_pub, test_priv = (d.merge(GEO, on="DIVIPOLA", how="left") for d in (train, test_pub, test_priv))


def agregar_vecindad(hist, filas):
    """Tasa histórica (global y del mes) de los 5 vecinos, con la MISMA historia que la fila (sin fuga)."""
    grilla = GEO[["DIVIPOLA"]].merge(MESES, how="cross")
    t = historial_congelado(hist, grilla)[["DIVIPOLA", "mes", "hist_tasa_mun", "hist_tasa_mes"]]
    v = (VECINOS.merge(t.rename(columns={"DIVIPOLA": "vecino"}), on="vecino")
                .groupby(["DIVIPOLA", "mes"])[["hist_tasa_mun", "hist_tasa_mes"]].mean()
                .rename(columns={"hist_tasa_mun": "hist_vecinos_tasa", "hist_tasa_mes": "hist_vecinos_tasa_mes"})
                .reset_index())
    return filas.merge(v, on=["DIVIPOLA", "mes"], how="left")


def historial_completo(hist, filas):
    return agregar_vecindad(hist, historial_congelado(hist, filas))


def historial_entrenamiento_completo(hist):
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_completo(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

Municipios del reto sin cartografía: 0 []


In [6]:
ONI_SERIE = pd.concat([train, test_pub, test_priv])[["fecha", "ONI"]].groupby("fecha")["ONI"].median()


ESPACIAL_COLS = ["lat", "lon", "log_area", "log_poblacion", "hist_vecinos_tasa", "hist_vecinos_tasa_mes"]


def features(df, espacial=False):
    X = pd.DataFrame(index=df.index)
    for c in COV_ANUALES:
        X["log_" + c] = np.log1p(df[c].clip(lower=0))
    X["ingresos_rango_anio"] = df.groupby("anio")["ingresos_tributarios"].rank(pct=True)
    for c in COV_ESTATICAS + ["ONI"] + HIST_COLS:
        X[c] = df[c]
    for c in [c for c in df.columns if c.endswith("_imputado")]:
        X[c] = df[c]
    for k in (1, 2, 3):
        X[f"ONI_rezago_{k}"] = (df["fecha"] - pd.DateOffset(months=k)).map(ONI_SERIE).to_numpy()
    X["mes"] = df["mes"]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)
    X["departamento"] = df["DIVIPOLA"].str[:2].astype(int)
    X["oni_x_inundacion"] = df["ONI"] * df["hist_pct_inund"]
    X["oni_x_tasa"] = df["ONI"] * df["hist_tasa_mun"]
    if espacial:
        X["lat"], X["lon"] = df["lat"], df["lon"]
        X["log_area"] = np.log1p(df["area_km2"])
        X["log_poblacion"] = np.log1p(df["densidad_poblacional"].clip(lower=0) * df["area_km2"])
        X["hist_vecinos_tasa"], X["hist_vecinos_tasa_mes"] = df["hist_vecinos_tasa"], df["hist_vecinos_tasa_mes"]
    return X

## 3. Métricas oficiales y validador de la plantilla

In [7]:
from sklearn.metrics import roc_auc_score

def metricas(pred, verdad):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y = np.log1p(m["personas_desplazadas"])
    est = np.log1p(m["personas_desplazadas_estimadas"])
    lo, hi = np.log1p(m["personas_desplazadas_q10"]), np.log1p(m["personas_desplazadas_q90"])
    wink = (hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)   # alpha = 0.2
    return {"auc": roc_auc_score(m["tiene_evento"], m["prob_evento"]),
            "rmse_log": float(np.sqrt(np.mean((y - est) ** 2))), "winkler": float(wink.mean())}

def puntos(met, nula):
    """Parte automática de la nota (máx. 75 puntos), como la tabla de posiciones."""
    s_auc = max(0, (met["auc"] - 0.5) / 0.5)
    s_rmse = min(1, max(0, 1 - met["rmse_log"] / nula["rmse_log"]))
    s_wink = min(1, max(0, 1 - met["winkler"] / nula["winkler"]))
    return 100 * (0.50 * s_auc + 0.20 * s_rmse + 0.05 * s_wink)


def prediccion_nula(filas):
    return filas[["DIVIPOLA", "fecha"]].assign(prob_evento=0.5, personas_desplazadas_estimadas=0.0,
                                               personas_desplazadas_q10=0.0, personas_desplazadas_q90=0.0)


def winkler_con_factor(pred, verdad, factor):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y, lo = np.log1p(m["personas_desplazadas"]), np.log1p(m["personas_desplazadas_q10"])
    hi = np.maximum(lo, np.log1p(m["personas_desplazadas_q90"]) * factor)
    return float(((hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)).mean())

In [8]:
COLUMNAS = ["DIVIPOLA", "fecha", "prob_evento", "personas_desplazadas_estimadas",
            "personas_desplazadas_q10", "personas_desplazadas_q90"]

def validar_entrega(e, test_pub=test_pub, test_priv=test_priv):
    errores = []
    faltan = [c for c in COLUMNAS if c not in e.columns]
    if faltan:
        return print("❌ Faltan columnas:", faltan)
    claves = pd.concat([test_pub, test_priv])[["DIVIPOLA", "fecha"]].assign(fecha=lambda d: pd.to_datetime(d.fecha))
    e2 = e.assign(DIVIPOLA=e.DIVIPOLA.astype(str).str.zfill(5), fecha=pd.to_datetime(e.fecha))
    if e2.duplicated(["DIVIPOLA", "fecha"]).any():
        errores.append("hay filas DIVIPOLA-fecha duplicadas")
    faltantes = len(claves.merge(e2, on=["DIVIPOLA", "fecha"], how="left", indicator=True).query("_merge == 'left_only'"))
    if faltantes:
        errores.append(f"faltan {faltantes} filas de las {len(claves)} esperadas")
    num = e2[COLUMNAS[2:]].apply(pd.to_numeric, errors="coerce")
    if num.isna().any().any():
        errores.append("hay valores vacíos o no numéricos")
    if not num.prob_evento.between(0, 1).all():
        errores.append("prob_evento debe estar entre 0 y 1")
    if (num[COLUMNAS[3:]] < 0).any().any():
        errores.append("las columnas de personas deben ser >= 0")
    if (num.personas_desplazadas_q10 > num.personas_desplazadas_q90).any():
        errores.append("personas_desplazadas_q10 debe ser <= personas_desplazadas_q90")
    print("✅ Entrega válida:", len(e), "filas" if not errores else "", *["\n❌ " + x for x in errores])
    return not errores

## 4. Variantes
`None` en el número de árboles significa early stopping (como la V1). Cada variante nueva cambia una cosa respecto a
la V1, salvo `E10_espacial_simple`.

In [9]:
import shutil
import time

import lightgbm as lgb

NOMBRE_EQUIPO = "AICOLOMBIA"
SEMILLA = 42
LGBM_BASE = dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, colsample_bytree=0.8, subsample=0.8,
                 subsample_freq=1, reg_lambda=1.0, random_state=SEMILLA, deterministic=True, force_row_wise=True,
                 verbose=-1)
ES_MESES, ES_PACIENCIA, FACTOR_ARBOLES, MAX_ARBOLES = 12, 100, 1.15, 3000

V1 = dict(arboles_p=None, arboles_mag=None, arboles_q=None, lgbm={}, peso_reciente=False, espacial=False)
SIMPLE = dict(arboles_p=100, lgbm=dict(num_leaves=15, min_child_samples=200))
VARIANTES = {
    "V1_early_stopping": {},
    "E1_arboles_200": dict(arboles_p=200, arboles_mag=300, arboles_q=300),
    "E4_hojas_15": dict(arboles_p=400, arboles_mag=300, arboles_q=300, lgbm=dict(num_leaves=15)),
    "E5_arboles_50": dict(arboles_p=50),
    "E5_arboles_100": dict(arboles_p=100),
    "E6_simple": SIMPLE,
    "E7_lento": dict(lgbm=dict(learning_rate=0.02)),
    "E8_peso_reciente": dict(peso_reciente=True),
    "E10_espacial": dict(espacial=True),
    "E10_espacial_simple": {**SIMPLE, "espacial": True},
}
DESCRIPCION = {
    "V1_early_stopping": "Versión 1 (early stopping)",
    "E1_arboles_200": "Ronda 1: clasificador 200 árboles",
    "E4_hojas_15": "Ronda 1: 400 árboles + 15 hojas",
    "E5_arboles_50": "V1 con clasificador de 50 árboles",
    "E5_arboles_100": "V1 con clasificador de 100 árboles",
    "E6_simple": "Clasificador 100 árboles, 15 hojas, mín. 200 por hoja",
    "E7_lento": "V1 con tasa de aprendizaje 0.02",
    "E8_peso_reciente": "V1 + más peso a los años recientes",
    "E10_espacial": "V1 + población, ubicación y vecinos",
    "E10_espacial_simple": "E6_simple + población, ubicación y vecinos",
    "E9_ensamble_top3": "Promedio de las 3 mejores variantes de la ronda 2",
}


def ajustar(X, y, fechas, clase, cfg, n_arboles, peso=None, **extra):
    params = {**LGBM_BASE, **cfg["lgbm"], **extra}
    if n_arboles is not None:
        return clase(**params, n_estimators=n_arboles).fit(X, y, sample_weight=peso), n_arboles
    meses = np.sort(fechas.unique())
    corte = meses[-min(ES_MESES, max(3, len(meses) // 4))]
    a, b = (fechas < corte).to_numpy(), (fechas >= corte).to_numpy()
    pa = None if peso is None else peso[a]
    m = clase(**params, n_estimators=MAX_ARBOLES).fit(X[a], y[a], sample_weight=pa, eval_set=[(X[b], y[b])],
                                                     callbacks=[lgb.early_stopping(ES_PACIENCIA, verbose=False)])
    n = max(50, int(round((m.best_iteration_ or MAX_ARBOLES) * FACTOR_ARBOLES)))
    return clase(**params, n_estimators=n).fit(X, y, sample_weight=peso), n


def entrenar(filas, cfg):
    X, fechas = features(filas, cfg["espacial"]), filas["fecha"]
    y_ev, y_log = filas["tiene_evento"], np.log1p(filas["personas_desplazadas"])
    peso = (1.25 ** (filas["anio"] - filas["anio"].min())).to_numpy() if cfg["peso_reciente"] else None
    modelos, arboles = {}, {}
    modelos["p"], arboles["p"] = ajustar(X, y_ev, fechas, lgb.LGBMClassifier, cfg, cfg["arboles_p"], peso,
                                         objective="binary", metric="auc")
    modelos["e"], arboles["e"] = ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_mag"], peso,
                                         objective="regression", metric="l2")
    for q in (0.1, 0.9):
        modelos[q], arboles[q] = ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_q"], peso,
                                         objective="quantile", alpha=q, metric="quantile")
    return modelos, arboles


def predecir_log(modelos, filas, cfg):
    """Predicciones crudas: probabilidad y magnitud/cuantiles en escala log1p (para poder promediar)."""
    X = features(filas, cfg["espacial"])
    p = filas[["DIVIPOLA", "fecha"]].copy()
    p["prob"] = modelos["p"].predict_proba(X)[:, 1]
    p["mag"] = np.clip(modelos["e"].predict(X), 0, None)
    p["lo"] = np.clip(modelos[0.1].predict(X), 0, None)
    p["hi"] = np.clip(modelos[0.9].predict(X), 0, None)
    return p


def a_entrega(cruda, factor_q90=1.0):
    p = cruda[["DIVIPOLA", "fecha"]].copy()
    p["prob_evento"] = cruda["prob"].to_numpy()
    p["personas_desplazadas_estimadas"] = np.expm1(cruda["mag"].to_numpy())
    hi = cruda["hi"].to_numpy() * factor_q90
    p["personas_desplazadas_q10"] = np.expm1(np.minimum(cruda["lo"].to_numpy(), hi))
    p["personas_desplazadas_q90"] = np.expm1(np.maximum(cruda["lo"].to_numpy(), hi))
    return p

## 5. Correr la ronda 2
El historial (con vecinos) de cada corte se calcula una sola vez. Por variante: validación F1–F3, factor de `q90`,
modelo final y CSV validado en `entregas_ronda2/`. Al final se arma el ensamble de las 3 mejores.

In [10]:
CORTES = [("F1", "2020-09-01", "2020-10-01", "2021-09-01"),
          ("F2", "2021-09-01", "2021-10-01", "2022-09-01"),
          ("F3", "2020-09-01", "2021-10-01", "2022-09-01")]
FACTORES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4]
CARPETA = "entregas_ronda2"

t0 = time.time()
PREP = {}
for nombre, fin_hist, ini_val, fin_val in CORTES:
    hist = train[train.fecha <= fin_hist]
    va = train[(train.fecha >= ini_val) & (train.fecha <= fin_val)]
    PREP[nombre] = (historial_entrenamiento_completo(hist), historial_completo(hist, va), va)
PRUEBA = pd.concat([test_pub, test_priv], ignore_index=True)
FINAL = (historial_entrenamiento_completo(train), historial_completo(train, PRUEBA))
print(f"Historial y vecindad preparados en {time.time() - t0:.0f}s")


def evaluar(crudas_por_corte):
    """Métricas por corte, factor de q90 y cobertura a partir de predicciones crudas."""
    por_corte, factores = {}, []
    for corte, cruda in crudas_por_corte.items():
        va = PREP[corte][2]
        pred = a_entrega(cruda)
        met, nula = metricas(pred, va), metricas(prediccion_nula(va), va)
        winklers = {f: winkler_con_factor(pred, va, f) for f in FACTORES}
        factores.append(min(winklers, key=winklers.get))
        m = va.merge(pred, on=["DIVIPOLA", "fecha"])
        cobertura = float(((m.personas_desplazadas >= m.personas_desplazadas_q10) &
                           (m.personas_desplazadas <= m.personas_desplazadas_q90)).mean())
        por_corte[corte] = {**met, "puntos": puntos(met, nula), "cobertura": cobertura}
    return pd.DataFrame(por_corte).T, float(np.median(factores))


def guardar(variante, cruda_final, factor):
    entrega = a_entrega(cruda_final, factor)
    entrega["fecha"] = pd.to_datetime(entrega["fecha"]).dt.strftime("%Y-%m-%d")
    entrega = entrega[COLUMNAS]
    if not validar_entrega(entrega):
        return "INVÁLIDA"
    archivo = f"{CARPETA}/entrega_{NOMBRE_EQUIPO}_{variante}.csv"
    entrega.to_csv(archivo, index=False)
    return archivo


os.makedirs(CARPETA, exist_ok=True)
CRUDAS, CRUDAS_FINAL, resumen, detalle = {}, {}, [], []


def registrar(variante, d, factor, archivo, segundos):
    resumen.append({"ronda": 2, "variante": variante, "descripcion": DESCRIPCION[variante],
                    "puntos_prom": d.puntos.mean(), "puntos_desv": d.puntos.std(),
                    "puntos_F1": d.loc["F1", "puntos"], "puntos_F2": d.loc["F2", "puntos"],
                    "puntos_F3": d.loc["F3", "puntos"], "auc_prom": d.auc.mean(), "rmse_log_prom": d.rmse_log.mean(),
                    "winkler_prom": d.winkler.mean(), "cobertura_prom": d.cobertura.mean(), "factor_q90": factor,
                    "archivo": archivo})
    for corte in d.index:
        detalle.append({"ronda": 2, "variante": variante, "corte": corte, **d.loc[corte].to_dict()})
    print(f"{variante:22s} puntos {d.puntos.mean():5.1f} ± {d.puntos.std():.1f} · F2 {d.loc['F2', 'puntos']:5.1f} "
          f"· F3 {d.loc['F3', 'puntos']:5.1f} · {segundos:.0f}s")


for variante, cambios in VARIANTES.items():
    t1 = time.time()
    cfg = {**V1, **cambios}
    CRUDAS[variante] = {}
    for corte, (filas_hist, filas_va, va) in PREP.items():
        modelos_cv, _ = entrenar(filas_hist, cfg)
        CRUDAS[variante][corte] = predecir_log(modelos_cv, filas_va, cfg)
    d, factor = evaluar(CRUDAS[variante])
    modelos_fin, _ = entrenar(FINAL[0], cfg)
    CRUDAS_FINAL[variante] = predecir_log(modelos_fin, FINAL[1], cfg)
    registrar(variante, d, factor, guardar(variante, CRUDAS_FINAL[variante], factor), time.time() - t1)

# Ensamble de las 3 mejores variantes de la ronda 2 (elegidas con los mismos cortes: optimismo leve)
top3 = pd.DataFrame(resumen).sort_values("puntos_prom", ascending=False)["variante"].head(3).tolist()
DESCRIPCION["E9_ensamble_top3"] += " (" + " + ".join(top3) + ")"
promedio = lambda partes: partes[0][["DIVIPOLA", "fecha"]].assign(
    **{c: np.mean([p[c].to_numpy() for p in partes], axis=0) for c in ["prob", "mag", "lo", "hi"]})
crudas_ens = {corte: promedio([CRUDAS[v][corte] for v in top3]) for corte in PREP}
d, factor = evaluar(crudas_ens)
registrar("E9_ensamble_top3", d, factor, guardar("E9_ensamble_top3", promedio([CRUDAS_FINAL[v] for v in top3]), factor), 0)
print(f"Total: {(time.time() - t0) / 60:.1f} min")

Historial y vecindad preparados en 3s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
V1_early_stopping      puntos  20.1 ± 3.9 · F2  24.1 · F3  16.2 · 20s
✅ Entrega válida: 42978 filas
E1_arboles_200         puntos  19.7 ± 3.4 · F2  22.8 · F3  16.0 · 11s
✅ Entrega válida: 42978 filas
E4_hojas_15            puntos  19.7 ± 3.5 · F2  23.1 · F3  16.1 · 12s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E5_arboles_50          puntos  21.1 ± 3.1 · F2  24.1 · F3  18.0 · 8s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

... (7 outputs left out)

c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10_espacial           puntos  21.5 ± 3.0 · F2  24.7 · F3  18.6 · 20s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10_espacial_simple    puntos  22.0 ± 2.8 · F2  24.9 · F3  19.3 · 8s
✅ Entrega válida: 42978 filas
E9_ensamble_top3       puntos  22.0 ± 2.8 · F2  24.9 · F3  19.2 · 0s
Total: 2.6 min


## 6. Resultados: ronda 1 + ronda 2

In [11]:
# Resultados de la ronda 1 (corrida en Colab): métricas por corte
RONDA1 = pd.DataFrame([
    ("COMBO", 0.679, 0.709, 0.652, 1.375, 1.269, 1.292, 3.452, 3.288, 3.446, 19.621, 22.632, 16.364),
    ("E1_arboles_200", 0.683, 0.712, 0.648, 1.364, 1.279, 1.289, 3.509, 3.269, 3.487, 20.163, 22.843, 16.035),
    ("E1_arboles_400", 0.679, 0.707, 0.648, 1.364, 1.279, 1.289, 3.509, 3.269, 3.487, 19.715, 22.324, 16.004),
    ("E1_arboles_800", 0.673, 0.701, 0.642, 1.364, 1.279, 1.289, 3.509, 3.269, 3.487, 19.142, 21.743, 15.339),
    ("E2_hurdle", 0.679, 0.707, 0.648, 1.376, 1.272, 1.294, 3.509, 3.269, 3.487, 19.542, 22.422, 15.935),
    ("E3_nivel_nacional", 0.680, 0.709, 0.651, 1.366, 1.280, 1.294, 3.479, 3.271, 3.464, 19.799, 22.510, 16.261),
    ("E4_hojas_15", 0.681, 0.714, 0.649, 1.361, 1.271, 1.288, 3.501, 3.296, 3.446, 19.922, 23.134, 16.147),
    ("E4_min_hoja_200", 0.679, 0.712, 0.648, 1.359, 1.274, 1.287, 3.459, 3.287, 3.469, 19.873, 22.845, 16.034),
    ("V1_early_stopping", 0.679, 0.722, 0.647, 1.353, 1.263, 1.268, 3.513, 3.248, 3.482, 19.872, 24.070, 16.250),
], columns=["variante", "auc_F1", "auc_F2", "auc_F3", "rmse_F1", "rmse_F2", "rmse_F3",
            "wink_F1", "wink_F2", "wink_F3", "puntos_F1", "puntos_F2", "puntos_F3"])
r1 = RONDA1.assign(ronda=1, descripcion="(ronda 1)", archivo="zip de la ronda 1",
                   puntos_prom=RONDA1[["puntos_F1", "puntos_F2", "puntos_F3"]].mean(axis=1),
                   puntos_desv=RONDA1[["puntos_F1", "puntos_F2", "puntos_F3"]].std(axis=1),
                   auc_prom=RONDA1[["auc_F1", "auc_F2", "auc_F3"]].mean(axis=1),
                   rmse_log_prom=RONDA1[["rmse_F1", "rmse_F2", "rmse_F3"]].mean(axis=1),
                   winkler_prom=RONDA1[["wink_F1", "wink_F2", "wink_F3"]].mean(axis=1))

r2 = pd.DataFrame(resumen)
repetidas = r2.merge(r1, on="variante", suffixes=("_r2", "_r1"))
for fila in repetidas.itertuples():
    igual = abs(fila.puntos_prom_r2 - fila.puntos_prom_r1) < 0.05
    print(f"Reproducibilidad {fila.variante}: ronda 1 {fila.puntos_prom_r1:.2f} vs ronda 2 {fila.puntos_prom_r2:.2f} "
          + ("✅" if igual else "⚠ difiere: normal si esta ronda corre en otro entorno (local vs. Colab) o con otra versión de LightGBM; para comparar, usar las filas de la ronda 2"))

todas = pd.concat([r2, r1[~r1.variante.isin(r2.variante)]], ignore_index=True)
v1 = r2.set_index("variante").loc["V1_early_stopping"]
todas["delta_vs_V1"] = todas["puntos_prom"] - v1["puntos_prom"]
todas["cumple_regla"] = (todas["delta_vs_V1"] >= 0.5) & (todas["puntos_F3"] >= v1["puntos_F3"])
tabla = todas.sort_values("puntos_prom", ascending=False).reset_index(drop=True)
columnas = ["ronda", "variante", "descripcion", "puntos_prom", "puntos_desv", "puntos_F1", "puntos_F2", "puntos_F3",
            "delta_vs_V1", "cumple_regla", "auc_prom", "rmse_log_prom", "winkler_prom", "cobertura_prom", "archivo"]
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
display(tabla[columnas].round(3))

print("Detalle por corte (ronda 2):")
display(pd.DataFrame(detalle).pivot(index="variante", columns="corte",
                                    values=["auc", "rmse_log", "winkler", "puntos"]).round(3))
tabla.to_csv(f"{CARPETA}/resultados_ronda1_y_2.csv", index=False)

candidatas = tabla[tabla["cumple_regla"] & (tabla["ronda"] == 2)]
if len(candidatas):
    mejor = candidatas.iloc[0]
    aviso = " · ⚠ usa cartografía: confirmar con los organizadores" if mejor.variante.startswith("E10") else ""
    print(f"Recomendada para entregar: {mejor.variante} ({mejor.descripcion}) · {mejor.puntos_prom:.1f} puntos "
          f"(+{mejor.delta_vs_V1:.1f} vs V1) · {mejor.archivo}{aviso}")
else:
    print("Ninguna variante cumple la regla (≥ 0.5 puntos sobre V1 sin empeorar F3): mantener la entrega V1.")

Reproducibilidad V1_early_stopping: ronda 1 20.06 vs ronda 2 20.06 ✅
Reproducibilidad E1_arboles_200: ronda 1 19.68 vs ronda 2 19.68 ✅
Reproducibilidad E4_hojas_15: ronda 1 19.73 vs ronda 2 19.73 ✅


,ronda,variante,descripcion,puntos_prom,puntos_desv,puntos_F1,puntos_F2,puntos_F3,delta_vs_V1,cumple_regla,auc_prom,rmse_log_prom,winkler_prom,cobertura_prom,archivo
0,2,E9_ensamble_top3,Promedio de las 3 mejores variantes de la ronda 2 (E10_e...,22.042,2.834,21.975,24.909,19.242,1.978,True,0.702,1.292,3.411,0.919,entregas_ronda2/entrega_AICOLOMBIA_E9_ensamble_top3.csv
1,2,E10_espacial_simple,"E6_simple + población, ubicación y vecinos",21.960,2.813,21.596,24.938,19.348,1.897,True,0.701,1.291,3.427,0.919,entregas_ronda2/entrega_AICOLOMBIA_E10_espacial_simple.csv
2,2,E7_lento,V1 con tasa de aprendizaje 0.02,21.651,2.774,22.057,24.199,18.697,1.587,True,0.698,1.293,3.412,0.917,entregas_ronda2/entrega_AICOLOMBIA_E7_lento.csv
3,2,E10_espacial,"V1 + población, ubicación y vecinos",21.525,3.031,21.282,24.670,18.622,1.461,True,0.697,1.294,3.413,0.920,entregas_ronda2/entrega_AICOLOMBIA_E10_espacial.csv
4,2,E6_simple,"Clasificador 100 árboles, 15 hojas, mín. 200 por hoja",21.231,2.781,21.299,23.977,18.416,1.167,True,0.694,1.292,3.412,0.919,entregas_ronda2/entrega_AICOLOMBIA_E6_simple.csv
5,2,E5_arboles_50,V1 con clasificador de 50 árboles,21.131,3.063,21.365,24.070,17.958,1.067,True,0.693,1.295,3.414,0.918,entregas_ronda2/entrega_AICOLOMBIA_E5_arboles_50.csv
6,2,E8_peso_reciente,V1 + más peso a los años recientes,21.049,3.117,21.069,24.156,17.922,0.985,True,0.693,1.297,3.419,0.919,entregas_ronda2/entrega_AICOLOMBIA_E8_peso_reciente.csv
7,2,E5_arboles_100,V1 con clasificador de 100 árboles,20.862,3.227,21.273,23.864,17.450,0.799,True,0.691,1.295,3.414,0.918,entregas_ronda2/entrega_AICOLOMBIA_E5_arboles_100.csv
8,2,V1_early_stopping,Versión 1 (early stopping),20.064,3.913,19.872,24.070,16.250,0.000,False,0.683,1.295,3.414,0.918,entregas_ronda2/entrega_AICOLOMBIA_V1_early_stopping.csv
9,2,E4_hojas_15,Ronda 1: 400 árboles + 15 hojas,19.734,3.497,19.922,23.134,16.147,-0.329,False,0.681,1.306,3.414,0.920,entregas_ronda2/entrega_AICOLOMBIA_E4_hojas_15.csv


Detalle por corte (ronda 2):


auc               rmse_log               winkler                puntos                
corte                   F1     F2     F3       F1     F2     F3      F1     F2     F3      F1      F2      F3
variante                                                                                                     
E10_espacial         0.693  0.728  0.671    1.353  1.261  1.267   3.520  3.251  3.468  21.282  24.670  18.622
E10_espacial_simple  0.696  0.731  0.677    1.351  1.259  1.263   3.511  3.316  3.454  21.596  24.938  19.348
E1_arboles_200       0.683  0.712  0.648    1.364  1.279  1.289   3.509  3.269  3.487  20.163  22.843  16.035
E4_hojas_15          0.681  0.714  0.649    1.361  1.271  1.288   3.501  3.296  3.446  19.922  23.134  16.147
E5_arboles_100       0.693  0.720  0.659    1.353  1.263  1.268   3.513  3.248  3.482  21.273  23.864  17.450
E5_arboles_50        0.694  0.722  0.664    1.353  1.263  1.268   3.513  3.248  3.482  21.365  24.070  17.958
E6_simple            0.693  0.721  0.668    1.351  1.259  1.265   3.500  3.295  3.441  21.299  23.977  18.416
E7_lento             0.701  0.723  0.672    1.351  1.260  1.268   3.499  3.262  3.475  22.057  24.199  18.697
E8_peso_reciente     0.691  0.723  0.665    1.353  1.267  1.271   3.510  3.258  3.487  21.069  24.156  17.922
E9_ensamble_top3     0.700  0.730  0.677    1.351  1.258  1.265   3.505  3.269  3.459  21.975  24.909  19.242
V1_early_stopping    0.679  0.722  0.647    1.353  1.263  1.268   3.513  3.248  3.482  19.872  24.070  16.250

Recomendada para entregar: E9_ensamble_top3 (Promedio de las 3 mejores variantes de la ronda 2 (E10_espacial_simple + E7_lento + E10_espacial)) · 22.0 puntos (+2.0 vs V1) · entregas_ronda2/entrega_AICOLOMBIA_E9_ensamble_top3.csv


## 7. Descargar los CSV de la ronda 2

In [12]:
shutil.make_archive("entregas_ronda2", "zip", CARPETA)
try:
    from google.colab import files
    files.download("entregas_ronda2.zip")
except ImportError:
    print("Guardado en", os.path.abspath("entregas_ronda2.zip"))

Guardado en g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\notebooks\entregas_ronda2.zip
